# Run Complete Pipeline (v3 - Downsampled Processing)

This notebook executes all pipeline stages in sequence.

**v3 Optimization**: All processing done at 2x downsampled resolution (~8x faster), then upsampled at the end.

**Stages:**
0. **Downsample** (~2 min) - Downsample labels and images by 2x
1. Hull Generation (~1 min)
2. Shrink-Wrapping (~1 min)
3. Major Axis Detection (~30 sec)
4. Socket Detection (~3 min total, was 4+ hours)
5. Bone Length Measurement (~30 sec)
6. **Upsample & Scale Metrics** (~1 min) - Restore full resolution
7. Excel Export (~1 min)
8. GIF Visualization (~5 min)

In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================

TARGET_DIR = "/mnt/c/users/mwild/firebase/perios/levi_data_1.6.26"

# Set to True to skip the slow socket detection step
SKIP_SOCKET_DETECTION = False

# ==========================================

In [ ]:
import time
from pathlib import Path
from datetime import timedelta

target = Path(TARGET_DIR)

print("="*70)
print("PHALANX MORPHOMETRIC ANALYSIS PIPELINE")
print("="*70)
print(f"\nTarget directory: {target}")
print(f"Skip socket detection: {SKIP_SOCKET_DETECTION}")

# Check input directories
labels_dir = target / "labels"
images_dir = target / "images"

if not labels_dir.exists():
    print(f"\nERROR: Labels directory not found: {labels_dir}")
else:
    n_labels = len(list(labels_dir.glob("*.nii.gz")))
    print(f"\nInput labels: {n_labels} files")

if images_dir.exists():
    n_images = len(list(images_dir.glob("*.nii.gz")))
    print(f"Input images: {n_images} files")

In [ ]:
# Define pipeline stages
stages = [
    ("00_downsample.ipynb", "Downsample Labels & Images", False),
    ("01_hull_generation.ipynb", "Hull Generation", False),
    ("02_shrink_wrap.ipynb", "Shrink-Wrapping", False),
    ("03_major_axis.ipynb", "Major Axis Detection", False),
    ("04_socket_detection.ipynb", "Socket Detection", False),  # Now fast due to downsampling!
    ("05_bone_length.ipynb", "Bone Length Measurement", False),
    ("06_upsample_metrics.ipynb", "Upsample & Scale Metrics", False),
    ("07_export_excel.ipynb", "Excel Export", False),
    ("08_visualization.ipynb", "GIF Visualization", False),
]

print("\nPipeline stages:")
for i, (notebook, name, is_slow) in enumerate(stages):
    slow_tag = " [SLOW]" if is_slow else ""
    skip_tag = " [SKIP]" if is_slow and SKIP_SOCKET_DETECTION else ""
    print(f"  {i}. {name}{slow_tag}{skip_tag}")

In [ ]:
# Run pipeline
import subprocess
import sys

total_start = time.time()
results = []

for i, (notebook, name, is_slow) in enumerate(stages, 1):
    print("\n" + "="*70)
    print(f"STAGE {i}: {name}")
    print("="*70)
    
    if is_slow and SKIP_SOCKET_DETECTION:
        print("SKIPPED (SKIP_SOCKET_DETECTION=True)")
        results.append((name, "skipped", 0))
        continue
    
    notebook_path = Path(notebook)
    if not notebook_path.exists():
        print(f"ERROR: Notebook not found: {notebook}")
        results.append((name, "error", 0))
        continue
    
    stage_start = time.time()
    
    try:
        # Execute notebook using jupyter nbconvert
        # First, inject the TARGET_DIR into the notebook
        cmd = [
            sys.executable, "-m", "jupyter", "nbconvert",
            "--to", "notebook",
            "--execute",
            "--inplace",
            "--ExecutePreprocessor.timeout=3600",
            str(notebook_path)
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        elapsed = time.time() - stage_start
        
        if result.returncode == 0:
            print(f"Completed in {timedelta(seconds=int(elapsed))}")
            results.append((name, "success", elapsed))
        else:
            print(f"FAILED after {timedelta(seconds=int(elapsed))}")
            print(f"Error: {result.stderr[:500]}...")
            results.append((name, "failed", elapsed))
            
    except Exception as e:
        elapsed = time.time() - stage_start
        print(f"ERROR: {e}")
        results.append((name, "error", elapsed))

total_elapsed = time.time() - total_start

In [ ]:
# Alternative: Run notebooks manually using %run magic
# Uncomment this cell and comment out the cell above to run interactively

# print("\n" + "="*70)
# print("STAGE 1: Hull Generation")
# print("="*70)
# %run 01_hull_generation.ipynb

# print("\n" + "="*70)
# print("STAGE 2: Shrink-Wrapping")
# print("="*70)
# %run 02_shrink_wrap.ipynb

# print("\n" + "="*70)
# print("STAGE 3: Major Axis Detection")
# print("="*70)
# %run 03_major_axis.ipynb

# if not SKIP_SOCKET_DETECTION:
#     print("\n" + "="*70)
#     print("STAGE 4: Socket Detection")
#     print("="*70)
#     %run 04_socket_detection.ipynb

# print("\n" + "="*70)
# print("STAGE 5: Bone Length Measurement")
# print("="*70)
# %run 05_bone_length.ipynb

# print("\n" + "="*70)
# print("STAGE 6: Excel Export")
# print("="*70)
# %run 06_export_excel.ipynb

In [ ]:
# Summary
print("\n" + "="*70)
print("PIPELINE COMPLETE")
print("="*70)

print(f"\nTotal time: {timedelta(seconds=int(total_elapsed))}")

print("\nResults:")
for name, status, elapsed in results:
    elapsed_str = str(timedelta(seconds=int(elapsed))) if elapsed > 0 else "-"
    status_emoji = {"success": "[OK]", "failed": "[FAIL]", "error": "[ERR]", "skipped": "[SKIP]"}[status]
    print(f"  {status_emoji} {name}: {elapsed_str}")

# Check outputs
output_file = target / "output.xlsx"
if output_file.exists():
    print(f"\nFinal output: {output_file}")
else:
    print(f"\nOutput file not found: {output_file}")